In [15]:
import json
import glob
import pandas as pd

## Finding the sleep files

sleep_files = sorted(glob.glob(r"C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\*sleepData.json"))

print(f"Found {len(sleep_files)} files:")
for f in sleep_files:
    print(f" {f}")

Found 9 files:
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\2023-09-11_2023-12-20_118214492_sleepData.json
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\2023-12-20_2024-03-29_118214492_sleepData.json
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\2024-03-29_2024-07-07_118214492_sleepData.json
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\2024-07-07_2024-10-15_118214492_sleepData.json
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\2024-10-15_2025-01-23_118214492_sleepData.json
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\2025-01-23_2025-05-03_118214492_sleepData.json
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\JSONs\DI_CONNECT\Sleep\2025-05-03_2025-08-11_118214492_sleepData.json
 C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning

In [16]:
## Collect all records into one list

all_records = []

for filepath in sleep_files:
    with open(filepath) as f:
        records = json.load(f)
    all_records.extend(records)

print(f"\nTotal records loaded: {len(all_records)}")


Total records loaded: 800


In [17]:
# Convert into a DataFrame
df = pd.DataFrame(all_records)

print(f"\nRaw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Raw shape: (800, 18)
Columns: ['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'calendarDate', 'sleepWindowConfirmationType', 'napList', 'deepSleepSeconds', 'lightSleepSeconds', 'remSleepSeconds', 'awakeSleepSeconds', 'unmeasurableSeconds', 'averageRespiration', 'lowestRespiration', 'highestRespiration', 'retro', 'awakeCount', 'avgSleepStress', 'sleepScores', 'restlessMomentCount']


In [18]:
# Flatten 'sleepScores' column

scores_df = pd.json_normalize(df["sleepScores"].dropna())
scores_df.index = df["sleepScores"].dropna().index
scores_df.columns = ["score_" + col for col in scores_df.columns]
 
df = df.drop(columns=["sleepScores"]).join(scores_df)
 
print(f"Shape after flattening sleepScores: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")


Shape after flattening sleepScores: (800, 31)

Columns:
['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'calendarDate', 'sleepWindowConfirmationType', 'napList', 'deepSleepSeconds', 'lightSleepSeconds', 'remSleepSeconds', 'awakeSleepSeconds', 'unmeasurableSeconds', 'averageRespiration', 'lowestRespiration', 'highestRespiration', 'retro', 'awakeCount', 'avgSleepStress', 'restlessMomentCount', 'score_overallScore', 'score_qualityScore', 'score_durationScore', 'score_recoveryScore', 'score_deepScore', 'score_remScore', 'score_lightScore', 'score_awakeningsCountScore', 'score_awakeTimeScore', 'score_combinedAwakeScore', 'score_restfulnessScore', 'score_interruptionsScore', 'score_feedback', 'score_insight']


In [19]:
# Sort by date and clean up 

df = df.sort_values("calendarDate").reset_index(drop=True)

print(f"\nDate range: {df['calendarDate'].iloc[0]}  →  {df['calendarDate'].iloc[-1]}")



Date range: 2023-12-11  →  nan


In [20]:
# Save to CSV
output_path = r"C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\sleep_concat.csv"
df.to_csv(output_path, index=False)
 
print(f"\nSaved to: {output_path}")


Saved to: C:\Users\maxk6\OneDrive\Desktop\DI-Connect-Wellness\Learning\sleep_concat.csv
